In [0]:
import pyspark.sql.functions as F
import pandas
from datetime import datetime, timedelta
from pyspark.sql.window import Window
import os
from marel.services import ServiceProvider
from marel.databricks import DatabricksProvider
from pyspark.ml.feature import StandardScaler
from pyspark.ml.functions import vector_to_array
import shap
import numpy as np
from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.sql.types import StructType, StructField

from pyspark.ml.classification import MultilayerPerceptronClassifier
import mlflow

provider = DatabricksProvider()
table_service = provider.table_service

In [0]:
%run /Users/valgerdur.valsdottir@marel.com/LIKX02/df_read_data_prodmedallion

In [0]:
# Read data using the shared function (112 features, no onTime)
days = 30
df_sx_data = df_read_data_prodmedallion(days)

In [0]:
os.environ["MLFLOW_DFS_TMP"] = "/Volumes/teams/sensorx/data-dump/mlflow_tmp"

# Load model (v7: 112 features,
#  15-day horizon)
model = mlflow.spark.load_model("models:/teams.sensorx.mlp_fault_prediction/7")
predictions = model.transform(df_sx_data).cache()

row_count = predictions.count()
print(f"Predictions for 15-day horizon: {row_count:,} rows")

## Output for decision makers

In [0]:
# --- MLP 15-day Model Inference: All machines + SHAP for top 5 ---
# Requires cells 1-4 (imports, %run function, read data, predict)

mlp_predictions = predictions

# Feature names (112 total: 5 sensor + 100 lags + 6 deviation + 1 deviceId, NO onTime)
sensor_cols = [
    "payload_xrayController_filamentCurrent",
    "payload_xrayController_temperature",
    "payload_xrayController_tubeCurrent",
    "payload_xrayController_tubeVoltage",
    "delta_seconds",
]
n_lags = 20
lag_cols = [f"{col}{L}" for L in range(1, n_lags + 1) for col in sensor_cols]
dev_sensors = [
    "payload_xrayController_filamentCurrent",
    "payload_xrayController_temperature",
    "payload_xrayController_tubeCurrent",
]
dev_cols = []
for col in dev_sensors:
    dev_cols.extend([f"{col}_avg_daily", f"{col}_dev_daily"])
feature_names = sensor_cols + lag_cols + dev_cols + ["deviceId_index"]
short_names = [n.replace("payload_xrayController_", "").replace("_avg_daily", "_avg").replace("_dev_daily", "_dev") for n in feature_names]

# --- Score ALL machines (last 24h, max prob per device) ---
w_rank = Window.partitionBy("properties_deviceID").orderBy(F.col("fault_probability").desc())

mlp_scored = mlp_predictions.select(
    "properties_payloadTimeStamp",
    "properties_deviceID",
    "features",
    F.round(vector_to_array("probability")[1], 4).alias("fault_probability"),
)

# One row per machine: highest risk score in last 24h
all_machines = (
    mlp_scored
    .filter(F.col("properties_payloadTimeStamp") >= F.current_timestamp() - F.expr("INTERVAL 24 HOURS"))
    .withColumn("rn", F.row_number().over(w_rank))
    .filter(F.col("rn") == 1)
    .drop("rn", "properties_payloadTimeStamp")
    .orderBy(F.col("fault_probability").desc())
)

# Display ALL machines with risk scores
print(f"{'='*70}")
print(f"ALL MACHINES - Risk Scores (15-day horizon, last 24h)")
print(f"{'='*70}\n")

all_machines_display = all_machines.select(
    F.col("properties_deviceID").alias("machine_id"),
    F.col("fault_probability").alias("risk_score")
)
display(all_machines_display)

# --- SHAP explanations for TOP 5 only ---
print(f"\n{'='*70}")
print(f"TOP 5 MACHINES - Detailed SHAP Explanations")
print(f"{'='*70}")

top_5_spark = all_machines.limit(5)

# Collect top 5 to driver
top_5_pd = top_5_spark.select(
    "properties_deviceID",
    vector_to_array("features").alias("features_array"),
    "fault_probability"
).toPandas()

X_explain = np.array(top_5_pd["features_array"].tolist())
device_ids = top_5_pd["properties_deviceID"].values
probs = top_5_pd["fault_probability"].values

# Background sample for SHAP (50 random rows)
background_pd = (
    mlp_predictions
    .select(vector_to_array("features").alias("features_array"))
    .sample(fraction=0.0001, seed=42)
    .limit(50)
    .toPandas()
)
X_background = np.array(background_pd["features_array"].tolist())

print(f"\nSHAP setup: explaining {len(X_explain)} devices against {len(X_background)} background samples")

# Predict function wrapping Spark MLP model
schema = StructType([StructField("features", VectorUDT(), True)])

def predict_fn_mlp15(X):
    rows = [(Vectors.dense(x.tolist()),) for x in X]
    df = spark.createDataFrame(rows, schema=schema)
    preds = model.transform(df).select(
        vector_to_array("probability")[1].alias("p")
    ).toPandas()
    return preds["p"].values

# Run SHAP KernelExplainer
explainer = shap.KernelExplainer(predict_fn_mlp15, X_background)
shap_values = explainer.shap_values(X_explain, nsamples=200)

# Build results with SHAP for top 5
results_rows = []
for i in range(len(X_explain)):
    sv = shap_values[i]
    top_3_idx = np.argsort(np.abs(sv))[-3:][::-1]
    
    print(f"\n#{i+1} Device: {device_ids[i]}")
    print(f"   Risk score: {probs[i]:.4f}")
    print(f"   Top 3 contributing features:")
    
    feature_contribs = []
    for rank, idx in enumerate(top_3_idx):
        direction = "+" if sv[idx] > 0 else "-"
        print(f"     {rank+1}. {short_names[idx]}: SHAP={sv[idx]:+.4f} (value={X_explain[i, idx]:.2f})")
        feature_contribs.append(f"{short_names[idx]} ({direction}{abs(sv[idx]):.3f})")
    
    results_rows.append((
        device_ids[i], float(probs[i]),
        feature_contribs[0], feature_contribs[1], feature_contribs[2]
    ))

results_df_15d = spark.createDataFrame(
    results_rows,
    ["machine_id", "fault_probability", "top_feature_1", "top_feature_2", "top_feature_3"]
)

print(f"\n")
display(results_df_15d)

In [0]:
import matplotlib.pyplot as plt

# Collect risk scores to driver for plotting
risk_scores = all_machines_display.select("risk_score").toPandas()

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(risk_scores["risk_score"], bins=30, edgecolor="black", alpha=0.7)
ax.set_xlabel("Risk Score (fault probability)")
ax.set_ylabel("Number of Machines")
ax.set_title("Distribution of Risk Scores Across All Machines (15-day horizon, last 24h)")
ax.axvline(risk_scores["risk_score"].median(), color="red", linestyle="--", label=f'Median: {risk_scores["risk_score"].median():.3f}')
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nSummary statistics:")
print(risk_scores["risk_score"].describe())